# Article text representations

BT4222 Group 6 · representations 1/3.

Fit **title TF-IDF + SVD on training news only**, freeze the transform, then encode title and abstract
in that same space. Export title, abstract and fused text vectors, article-row mappings and title token
sequences for the NRMS owner. This is a reproducible compact baseline, not a trained NRMS model.

Run top to bottom using the same Python environment as the other notebooks. All project code and settings are included below; no local Python module or settings file is imported. Upstream Parquet/NumPy artifacts remain the inputs shared between notebooks.

`REBUILD=False` reuses verified saved outputs; set it to `True` after changing this notebook's calculations/settings. Restarting VS Code clears variables, not saved datasets.

In [ ]:
# All project settings and implementation are in this notebook.
# Standard third-party packages come from requirements.txt.
import os
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

try:
    import google.colab
    from google.colab import drive

    drive.mount("/content/drive")
except ImportError:
    pass

# Edit DATA_ROOT to point to another project/data folder, if necessary.
DATA_ROOT = None
candidates = [Path.cwd(), *Path.cwd().parents, Path("/content/drive/MyDrive/BT4222")]
if DATA_ROOT is not None:
    ROOT = Path(DATA_ROOT).expanduser().resolve()
elif os.environ.get("NEWSREC_ROOT"):
    ROOT = Path(os.environ["NEWSREC_ROOT"]).expanduser().resolve()
else:
    ROOT = next(
        (
            p
            for p in candidates
            if (p / "1_Data_Preparation/processed/news_catalogue.parquet").is_file()
        ),
        None,
    )
if ROOT is None:
    raise FileNotFoundError("Run data preparation first, or set DATA_ROOT above.")
ARTIFACTS = ROOT / "3_Representations/artifacts"
print("Data and output root:", ROOT)

In [ ]:
import hashlib
import importlib.metadata
import json
import platform
import subprocess
import time
from contextlib import contextmanager
from dataclasses import asdict, dataclass
from pathlib import Path
import numpy as np
import pandas as pd
import psutil
import pickle
from collections import Counter
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from dataclasses import dataclass

### Artifact checks and reporting

In [ ]:
NOTEBOOK_PATHS = (
    "3_Representations/news_embeddings.ipynb",
    "3_Representations/entity_embeddings.ipynb",
    "3_Representations/user_profiles.ipynb",
    "4_Feature_Engineering/build_ranking_features.ipynb",
)

In [ ]:
SPLITS = ("train", "validation", "test")

In [ ]:
VERSION = "newsrec-v1"

In [ ]:
@dataclass(frozen=True)
class Settings:
    seed: int = 4222
    text_dim: int = 64
    max_features: int = 50000
    min_df: int = 2
    title_weight: float = 0.7
    title_tokens: int = 30
    entity_dim: int = 100
    history_cap: int = 50
    snapshot_order_verified: bool = False
    recent_count: int = 10
    recent_hours: int = 24
    interest_count: int = 3
    ctr_strength: float = 20.0
    half_life_min_exposures: int = 100
    negative_count: int = 4
    update_mode: str = "prequential"
    context_features: bool = False
    parquet_batch_rows: int = 100000
    output_batch_rows: int = 100000

    def __post_init__(self):
        for key in (
            "text_dim",
            "max_features",
            "min_df",
            "title_tokens",
            "entity_dim",
            "history_cap",
            "recent_count",
            "recent_hours",
            "interest_count",
            "half_life_min_exposures",
            "negative_count",
            "parquet_batch_rows",
            "output_batch_rows",
        ):
            if getattr(self, key) <= 0:
                raise ValueError(f"{key} must be positive")
        if not 0 <= self.title_weight <= 1 or self.ctr_strength <= 0:
            raise ValueError("Invalid fusion/smoothing weights")
        if self.update_mode not in ("prequential", "frozen"):
            raise ValueError("update_mode must be prequential or frozen")

    def representation_fingerprint(self):
        fields = (
            "seed",
            "text_dim",
            "max_features",
            "min_df",
            "title_weight",
            "title_tokens",
            "entity_dim",
        )
        values = {name: getattr(self, name) for name in fields}
        return hashlib.sha256(json.dumps(values, sort_keys=True).encode()).hexdigest()

    def fingerprint(self):
        return hashlib.sha256(
            json.dumps(asdict(self), sort_keys=True).encode()
        ).hexdigest()

In [ ]:
DEFAULT_SETTINGS = Settings()

In [ ]:
# Edit settings here. Keep representation dimensions/vocabulary settings
# consistent across the notebooks that consume those artifacts.
SETTINGS = Settings(
    seed=4222,
    text_dim=64,
    max_features=50000,
    min_df=2,
    title_weight=0.7,
    title_tokens=30,
    entity_dim=100,
    history_cap=50,
    snapshot_order_verified=False,
    recent_count=10,
    recent_hours=24,
    interest_count=3,
    ctr_strength=20.0,
    half_life_min_exposures=100,
    negative_count=4,
    update_mode="prequential",
    context_features=False,
    parquet_batch_rows=100000,
    output_batch_rows=100000,
)
REBUILD = (
    False  # False: reuse verified outputs; True: regenerate this notebook's outputs
)

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

In [ ]:
def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, default=str) + "\n")

In [ ]:
def normalise(values):
    values = np.asarray(values, dtype=np.float32)
    norms = np.linalg.norm(values, axis=-1, keepdims=True)
    return np.divide(values, norms, out=np.zeros_like(values), where=norms > 1e-12)

In [ ]:
def manifest(root, cfg, sources, outputs, **extra):
    root = Path(root)
    try:
        revision = subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=root, stderr=subprocess.DEVNULL, text=True
        ).strip()
    except (subprocess.CalledProcessError, FileNotFoundError):
        revision = None
    return dict(
        version=VERSION,
        created_utc=pd.Timestamp.now(tz="UTC").isoformat(),
        config=asdict(cfg),
        config_hash=cfg.fingerprint(),
        representation_config_hash=cfg.representation_fingerprint(),
        git_revision=revision,
        python=platform.python_version(),
        libraries={
            n: importlib.metadata.version(n)
            for n in ("numpy", "pandas", "pyarrow", "polars", "scikit-learn")
        },
        implementation_hashes={
            relative: sha256(root / relative)
            for relative in NOTEBOOK_PATHS
            if (root / relative).is_file()
        },
        sources={str(Path(p).relative_to(root)): sha256(p) for p in sources},
        outputs={str(Path(p).relative_to(root)): sha256(p) for p in outputs},
        output_bytes=sum((Path(p).stat().st_size for p in outputs)),
        **extra,
    )

In [ ]:
@contextmanager
def measure(root, stage, **details):
    import threading

    process = psutil.Process()
    peak = [process.memory_info().rss]
    stop = threading.Event()

    def monitor():
        while not stop.wait(0.05):
            peak[0] = max(peak[0], process.memory_info().rss)

    worker = threading.Thread(target=monitor, daemon=True)
    worker.start()
    start = time.perf_counter()
    try:
        yield
    finally:
        peak[0] = max(peak[0], process.memory_info().rss)
        stop.set()
        worker.join()
        path = Path(root) / "results" / "representation_cost.csv"
        path.parent.mkdir(parents=True, exist_ok=True)
        row = dict(
            stage=stage,
            wall_seconds=time.perf_counter() - start,
            sampled_peak_process_rss_mb=peak[0] / 1000000.0,
            platform=platform.platform(),
            cpu=platform.processor() or platform.machine(),
            logical_cpus=psutil.cpu_count(),
            ram_gb=psutil.virtual_memory().total / 1000000000.0,
            **details,
        )
        pd.DataFrame([row]).to_csv(
            path, mode="a", header=not path.exists(), index=False
        )

In [ ]:
def upsert_report(root, filename, frame, stage):
    path = Path(root) / "results" / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    frame = frame.copy()
    frame["stage"] = stage
    if path.exists():
        old = pd.read_csv(path)
        frame = pd.concat([old[old.stage != stage], frame], ignore_index=True)
    frame.to_csv(path, index=False)

In [ ]:
def require_inputs(root):
    p = Path(root) / "1_Data_Preparation" / "processed"
    files = [p / "news_catalogue.parquet", p / "users.parquet"]
    files += [
        p / f"{kind}_{split}.parquet"
        for kind in ("impressions", "candidates")
        for split in SPLITS
    ]
    missing = [str(f) for f in files if not f.exists()]
    if missing:
        raise FileNotFoundError(
            "Run 1_Data_Preparation/clean_and_validate_data.ipynb first. Missing:\n"
            + "\n".join(missing)
        )
    return p

### Article representation and availability logic

In [ ]:
STATIC_COLUMNS = [
    "news_id",
    "news_code",
    "category",
    "subcategory",
    "title",
    "abstract",
    "title_entities",
    "abstract_entities",
]

In [ ]:
def artifact_dir(root):
    path = Path(root) / "3_Representations" / "artifacts"
    path.mkdir(parents=True, exist_ok=True)
    return path

In [ ]:
def clean_text(value):
    import unicodedata

    return " ".join(
        unicodedata.normalize("NFKC", "" if pd.isna(value) else str(value)).split()
    )

In [ ]:
def read_catalogue(root):
    path = Path(root) / "1_Data_Preparation" / "processed" / "news_catalogue.parquet"
    df = pd.read_parquet(path)
    required = STATIC_COLUMNS + ["in_news_train"]
    if not set(required) <= set(df):
        raise ValueError(f"Catalogue lacks {set(required) - set(df)}")
    df = df.sort_values("news_code").reset_index(drop=True)
    if df.news_code.duplicated().any() or df.news_id.duplicated().any():
        raise ValueError("Article IDs must be unique")
    return df

In [ ]:
def audit_metadata_conflicts(root):
    """Compare raw split metadata when available; preparation keeps first occurrence."""
    pieces = []
    for split in SPLITS:
        p = Path(root) / split / f"news_{split}.parquet"
        if p.exists():
            frame = pd.read_parquet(p, columns=STATIC_COLUMNS[:1] + STATIC_COLUMNS[2:])
            frame["source"] = split
            pieces.append(frame)
    if len(pieces) < 2:
        return {"status": "raw metadata not available for cross-release comparison"}
    joined = pd.concat(pieces, ignore_index=True).fillna("")
    conflicts = {
        c: int((joined.groupby("news_id")[c].nunique() > 1).sum())
        for c in STATIC_COLUMNS[2:]
    }
    return {
        "status": "checked; prepared catalogue retains deterministic source precedence",
        "conflicting_article_counts": conflicts,
    }

In [ ]:
def build_news(root, cfg=DEFAULT_SETTINGS):
    processed = require_inputs(root)
    out = artifact_dir(root)
    with measure(root, "text"):
        news = read_catalogue(root)
        titles = news.title.map(clean_text)
        abstracts = news.abstract.map(clean_text)
        train = news.in_news_train.fillna(False).to_numpy(dtype=bool)
        if train.sum() < 2:
            raise ValueError("At least two training articles are required")
        tfidf = TfidfVectorizer(
            ngram_range=(1, 2),
            max_features=cfg.max_features,
            min_df=cfg.min_df,
            dtype=np.float32,
        )
        train_tfidf = tfidf.fit_transform(titles[train])
        components = min(
            cfg.text_dim, train_tfidf.shape[0] - 1, train_tfidf.shape[1] - 1
        )
        if components < 1:
            raise ValueError(
                "Insufficient training vocabulary for SVD; inspect min_df/text"
            )
        svd = TruncatedSVD(n_components=components, random_state=cfg.seed)
        svd.fit(train_tfidf)
        matrices = []
        for texts in (titles, abstracts):
            reduced = normalise(svd.transform(tfidf.transform(texts)))
            dense = np.zeros((len(news) + 1, cfg.text_dim), np.float32)
            dense[1:, :components] = reduced
            matrices.append(dense)
        title, abstract = matrices
        tv = np.linalg.norm(title, axis=1) > 0
        av = np.linalg.norm(abstract, axis=1) > 0
        text = normalise(cfg.title_weight * title + (1 - cfg.title_weight) * abstract)
        text[tv & ~av] = title[tv & ~av]
        text[av & ~tv] = abstract[av & ~tv]
        index = news[STATIC_COLUMNS].copy()
        index.insert(0, "article_row", np.arange(1, len(news) + 1, dtype=np.int32))
        for col in ("category", "subcategory"):
            vocabulary = sorted(set(news.loc[train, col].dropna().astype(str)))
            lookup = {v: i + 1 for i, v in enumerate(vocabulary)}
            index[f"{col}_code"] = news[col].map(lookup).fillna(0).astype(np.int32)
            write_json(out / f"{col}_vocabulary.json", lookup)
        index["has_abstract"] = abstracts.ne("")
        index["title_word_count"] = titles.str.split().str.len().astype(np.int32)
        index["abstract_word_count"] = abstracts.str.split().str.len().astype(np.int32)
        index["has_title_vector"] = tv[1:]
        index["has_abstract_vector"] = av[1:]
        index["has_text_vector"] = np.linalg.norm(text[1:], axis=1) > 0
        index.to_parquet(out / "article_index.parquet", index=False)
        for name, matrix in (
            ("title64", title),
            ("abstract64", abstract),
            ("text64", text),
        ):
            np.save(out / f"{name}.npy", matrix)
        with (out / "text_transform.pkl").open("wb") as fh:
            pickle.dump({"tfidf": tfidf, "svd": svd}, fh)
        tokenise = TfidfVectorizer().build_analyzer()
        frequency = Counter(
            (token for text_ in titles[train] for token in tokenise(text_))
        )
        token_vocab = {
            w: i + 2
            for i, (w, _) in enumerate(
                sorted(frequency.items(), key=lambda x: (-x[1], x[0]))[
                    : cfg.max_features
                ]
            )
        }
        token_ids = np.zeros((len(news) + 1, cfg.title_tokens), np.int32)
        token_lengths, unknown, total = ([], 0, 0)
        for i, text_ in enumerate(titles, 1):
            tokens = tokenise(text_)
            token_lengths.append(len(tokens))
            ids = [token_vocab.get(w, 1) for w in tokens]
            unknown += sum((x == 1 for x in ids))
            total += len(ids)
            token_ids[i, : min(len(ids), cfg.title_tokens)] = ids[: cfg.title_tokens]
        np.save(out / "title_token_ids.npy", token_ids)
        np.save(out / "title_token_mask.npy", token_ids != 0)
        write_json(
            out / "title_vocabulary.json",
            {"padding_id": 0, "unknown_id": 1, "tokens": token_vocab},
        )
        coverage = []
        for split in SPLITS:
            mask = news[f"in_news_{split}"].to_numpy(bool)
            for rep, valid in (
                ("title", tv[1:]),
                ("abstract", av[1:]),
                ("text", index.has_text_vector.to_numpy()),
            ):
                n = int(mask.sum())
                k = int(valid[mask].sum())
                coverage.append(
                    {
                        "split": split,
                        "representation": rep,
                        "cohort": "split_catalogue",
                        "denominator": n,
                        "valid": k,
                        "fallback": n - k,
                        "valid_fraction": k / n if n else 0,
                    }
                )
        upsert_report(
            root, "representation_coverage.csv", pd.DataFrame(coverage), "text"
        )
        outputs = [
            out / f"{name}.npy"
            for name in (
                "title64",
                "abstract64",
                "text64",
                "title_token_ids",
                "title_token_mask",
            )
        ] + [out / "article_index.parquet", out / "text_transform.pkl"]
        outputs += [
            out / f"{c}_vocabulary.json" for c in ("title", "category", "subcategory")
        ]
        write_json(
            out / "news_manifest.json",
            manifest(
                root,
                cfg,
                [processed / "news_catalogue.parquet"],
                outputs,
                padded_row=0,
                effective_svd_dim=components,
                stored_text_dim=cfg.text_dim,
                article_index_hash=sha256(out / "article_index.parquet"),
                fit_scope="unique training-release article titles only",
                token_oov_fraction=unknown / max(1, total),
                title_truncation_fraction=float(
                    np.mean(np.array(token_lengths) > cfg.title_tokens)
                ),
                metadata_audit=audit_metadata_conflicts(root),
            ),
        )
    return pd.DataFrame(coverage)

In [ ]:
def saved_stage_ready(root, settings, stage):
    """Reuse disk outputs only when settings, inputs and output hashes agree."""
    path = Path(root) / "3_Representations/artifacts" / f"{stage}_manifest.json"
    if not path.exists():
        return False
    meta = json.loads(path.read_text())
    key = "config_hash" if stage == "profiles" else "representation_config_hash"
    expected = (
        settings.fingerprint()
        if stage == "profiles"
        else settings.representation_fingerprint()
    )
    if meta[key] != expected:
        raise ValueError(
            "Saved settings differ. Set REBUILD=True and rerun dependent notebooks."
        )
    for kind in ("sources", "outputs"):
        for relative, expected_hash in meta[kind].items():
            file = Path(root) / relative
            if not file.exists() or sha256(file) != expected_hash:
                raise ValueError(
                    f"Saved {stage} artifacts are stale or incomplete: {relative}. Set REBUILD=True."
                )
    return True

## Build or load saved outputs

## 1. Fit and export

Defaults: title unigrams/bigrams, up to 50,000 terms, `min_df=2`, SVD64, fixed seed.
All vectors are float32 and L2-normalised when valid. The combined view weights title/abstract 0.7/0.3;
a missing/OOV field falls back to the other valid field. Completely unsupported text remains zero with a mask.
Array row 0 is reserved for padding; real `news_code` values must go through `article_index.parquet`.
Training-fitted category/subcategory vocabularies reserve code 0 for unseen values.

Rebuilding representations invalidates dependent artifacts. Use one settings across the three notebooks.

In [ ]:
if REBUILD or not saved_stage_ready(ROOT, SETTINGS, "news"):
    coverage = build_news(ROOT, SETTINGS)
else:
    print("Loaded saved article text representations.")
    report = pd.read_csv(ROOT / "results/representation_coverage.csv")
    coverage = report[report.stage.eq("text")]
display(coverage)

## 2. Check dimensions, norms and quality

A present title can become a zero vector after train-vocabulary filtering. Abstract absence and zero-vector
rates are distinct. On a tiny fixture the effective SVD rank can be smaller than the configured dimension;
unused dimensions are padded with zeros and the effective rank is recorded.

In [ ]:
index = pd.read_parquet(ARTIFACTS / "article_index.parquet")
summary = []
for name in ["title64", "abstract64", "text64"]:
    array = np.load(ARTIFACTS / f"{name}.npy", mmap_mode="r")
    norms = np.linalg.norm(array, axis=1)
    assert array.shape == (len(index) + 1, SETTINGS.text_dim)
    assert np.isfinite(array).all() and np.all(array[0] == 0)
    assert np.allclose(norms[norms > 1e-7], 1, atol=1e-4)
    summary.append(
        dict(
            representation=name,
            shape=str(array.shape),
            MB=array.nbytes / 1e6,
            valid_articles=int((norms[1:] > 0).sum()),
        )
    )
display(pd.DataFrame(summary))
display(
    index[
        [
            "news_code",
            "title",
            "has_abstract",
            "has_title_vector",
            "has_abstract_vector",
        ]
    ].head()
)
meta = json.loads((ARTIFACTS / "news_manifest.json").read_text())
display(
    {
        k: meta[k]
        for k in [
            "effective_svd_dim",
            "stored_text_dim",
            "fit_scope",
            "token_oov_fraction",
            "title_truncation_fraction",
            "metadata_audit",
        ]
    }
)

## 3. Inspect neighbours without using click labels

This is a qualitative sanity check of shared text coordinates, not evidence of a ranking gain.
The frozen vocabulary can leave new articles unsupported; the serving pipeline retains those articles
and uses entity/popularity/freshness fallback rather than dropping them.

In [ ]:
vectors = np.load(ARTIFACTS / "text64.npy", mmap_mode="r")
valid = np.flatnonzero(np.linalg.norm(vectors, axis=1) > 0)
if len(valid):
    query = int(valid[0])
    scores = vectors @ vectors[query]
    candidates = valid[valid != query]
    neighbours = candidates[np.argsort(-scores[candidates], kind="stable")[:5]]
    display(
        index.set_index("article_row").loc[[query, *neighbours], ["news_code", "title"]]
    )
    print("Neighbour cosine scores:", scores[neighbours])
display(pd.read_csv(ROOT / "results/representation_cost.csv").tail())

## Handoff

Consumers can load the saved arrays with NumPy and the article index with pandas after all three notebooks finish.
`title64` reproduces the report's multi-interest space; `text64` is a separate combined-content view.
NRMS receives `title_token_ids.npy`, its mask, and the training-only vocabulary (`0=PAD`, `1=UNK`);
its attention encoders and learned embeddings belong to the NRMS notebook.